# Case Study §2 — Data Availability: why SFT data is the bottleneck

Runnable twin of [`02_data_availability.py`](02_data_availability.py). CPT and SFT need different data,
and they are **not** equally easy to get:
- **CPT** wants **raw text** — *abundant* (§1 scraped ~100k tokens in seconds, unlabelled).
- **SFT** wants **instruction pairs** (prompt→completion) — *scarce* (every answer is authored by hand).

And the two shapes map directly to two losses (RESEARCH_NOTES.md §7):
- corpus → `{"text": ...}` → **full causal loss** (predict every token)
- SFT → `{"prompt":…, "completion":…}` → **completion-only loss** (prompt masked)

In [ ]:
import importlib.util, pathlib, sys, json
HERE = pathlib.Path.cwd()
scripts_dir = next(p for p in [HERE/'scripts', HERE.parent/'scripts', HERE] if (p/'config.py').exists())
script = scripts_dir / "02_data_availability.py"
sys.path.insert(0, str(script.parent))
spec = importlib.util.spec_from_file_location("data_avail", script)
da = importlib.util.module_from_spec(spec); spec.loader.exec_module(da)

In [ ]:
stats = da.analyze()

### A few of the hand-authored SFT pairs
These are the *expensive* data: factual, neutral, written and checked by hand. Note how few there are —
that scarcity is the whole point.

In [ ]:
for r in da.SEED_QA[:4]:
    print('Q:', r['prompt'])
    print('A:', r['completion'])
    print()

# Verify shapes + the asymmetry the chapter relies on
assert stats['corpus_tokens'] > 20 * stats['sft_tokens'], 'corpus should dwarf the SFT set'
assert stats['sft_pairs'] < 100, 'hand-built SFT data is deliberately small'
rows = [json.loads(l) for l in open(da.config.SFT_DIR / 'seed_qa.jsonl')]
assert all('prompt' in r and 'completion' in r for r in rows), 'prompt-completion shape'
print(f"\u2713 verified: {stats['sft_pairs']} SFT pairs vs ~{stats['ratio']:.0f}\u00d7 more corpus tokens; prompt-completion shape OK")

**Takeaway:** raw domain text is ~100× more plentiful than instruction pairs, and far cheaper to obtain.
So the realistic question for a niche domain is *small-data* SFT — which is exactly why §6 tests whether to
start fine-tuning from the **base** or the **instruct** model when only a handful of examples exist.

**Next:** `03_cpt.py` — continued pretraining on the corpus (full causal loss, LoRA incl. `embed_tokens`/`lm_head`).